In [ ]:
using ProgressBars
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

In [ ]:
ω_a = 25.51325

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####
#ω_a = 3.85*(rand(1)[1]/5+0.9)*2*pi
ω_b_0 = 2*ω_a

Lamb_shift = 0.05 #Lamb shift in GHz


φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b_0
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.5


##Drive


ω_d=ω_b_0#+Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d

ϵ_0 = 0.61
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d = 10.12*g2


buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
#κ_2 = 4*g2^2/κ_b
κ_a = 0.0;#3e-3*κ_2 #from Leon's paper

In [ ]:
t_max = 250/κ_b
t_min = 0.0
t_min_save = t_max - 10/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(ω_a)
t_save = max(t_min_save,0):dt_save:t_max

b_0=0.0

environment = MeanField(κ_a, κ_b)
environment_prepared = environment

drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
drive_prepared = prepare(drive)

t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

In [ ]:
Im_a_list=range(-4,4,10)
Re_a_list=range(-5,5,25)

Final_mode_a = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
Final_mode_b = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
for i in 1:length(Re_a_list)
    re_i = Re_a_list[i]
    for j in 1:length(Im_a_list)
        im_j = Im_a_list[j]
        a_0_j = re_i + 1im*im_j

        system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0_j, b_0)
        system_prepared = prepare(system)

        u_0 = get_init(system_prepared, environment_prepared)
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

        Final_mode_a[(re_i,im_j)] = a
        Final_mode_b[(re_i,im_j)] = b
    end
end

In [ ]:
function plot_mode(Final_mode)
    p0 = plot()

    # Overlay scatter points
    for (re_i, im_j) in keys(Final_mode)
        z = Final_mode[(re_i, im_j)]
        idx_even = 2:2:length(z)
        idx_odd  = 1:2:length(z)
        scatter!(
            p0,
            real.(z[idx_even]),
            imag.(z[idx_even]),
            ms = 1,
            color = :blue,
            label = "",
        )

        scatter!(
            p0,
            real.(z[idx_odd]),
            imag.(z[idx_odd]),
            ms = 1,
            color = :red,
            label = "",
        )
    end

    # Add labels and title. 
    plot!(p0,
            xlabel=L"\Re(a)",
            ylabel=L"\Im(a)",
            title=L"\kappa_a="*string(κ_a)*" GHz",
            top_margin=5mm)
    return p0
end

In [ ]:
p0 = plot_mode(Final_mode_a)
display(p0)


In [ ]:
p0 = plot_mode(Final_mode_b)
display(p0)

# Dichotomy

In [ ]:
function orbit_cardinality(z::AbstractVector{<:Complex}; eps=1e-2)
    reps = ComplexF64[]
    for zi in z
        if all(abs(zi - r) > eps for r in reps)
            push!(reps, zi)
        end
    end
    return length(reps)
end
function run_notebook()
    Final_mode_a = Vector{ComplexF32}()
    for i in 1:length(Re_a_list)
        re_i = Re_a_list[i]
        for j in 1:length(Im_a_list)
            im_j = Im_a_list[j]
            a_0_j = re_i + 1im*im_j

            system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0_j, b_0)
            system_prepared = prepare(system)

            u_0 = get_init(system_prepared, environment_prepared)
            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            a, _, _ = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

            append!(Final_mode_a, a)
        end
    end
    return Final_mode_a
end


In [ ]:
t_max = 500/κ_b
t_min = 0.0
t_min_save = t_max - 10/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(ω_a)
t_save = max(t_min_save,0):dt_save:t_max

b_0=0.0

drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
drive_prepared = prepare(drive)

t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

In [ ]:
kappa_a_min = 0.0
kappa_a_max = 1

gap = 1e-4
res_dict = Dict{Float64, Float64}()
Final_mode_b = Vector{ComplexF64}()
while abs(kappa_a_max - kappa_a_min) > gap
    κ_a = (kappa_a_min + kappa_a_max) / 2
    
    environment = MeanField(κ_a, κ_b)
    environment_prepared = environment
    
    Final_mode_b = run_notebook()
    res = orbit_cardinality(Final_mode_b)
    res_dict[κ_a] = res
    if  (res == 4) || (res == 3) || (res == 2)
        kappa_a_max = κ_a
    elseif (res == 6) || (res == 10)
        break
    elseif res > 10
        kappa_a_min = κ_a
    else 
        break
    end
end
print(res_dict)

In [ ]:
(kappa_a_max, kappa_a_min)

In [ ]:
res_dict[kappa_a_max]

In [ ]:
res_dict[kappa_a_min]